In [ ]:
import csv
import random
import torch
from pymatgen.core import Structure
from torch_geometric.nn import CGConv

In [66]:
DATA_FILE = "data/data.csv"
TEST_FORMULAS = ["H3Se","CaH6"]
RADIUS = 3.0              # distance to neighbors
ITERATIONS = 200

random.seed(0)
torch.manual_seed(0)

CENTERS = torch.linspace(0.5, 3.0, 26)

In [ ]:
import os

for row in csv.DictReader(open(DATA_FILE)):
    exists = "ok" if os.path.exists(row["cif_file"]) else "FILE NOT FOUND"
    print(row["formula"], row["pressure_GPa"], "GPa,", row["Tc_K"], "K ->", row["cif_file"], f"({exists})")

In [67]:
def read_crystal(cif_file):
    structure = Structure.from_file(cif_file).get_primitive_structure()

    atoms = []
    for site in structure:
        atoms.append(site.specie.Z)

    sources = []
    targets = []
    distances = []
    for i in range(len(structure)):
        for neighbor in structure.get_neighbors(structure[i], RADIUS):
            sources.append(neighbor.index)
            targets.append(i)
            distances.append(neighbor.nn_distance)

    d = torch.tensor(distances, dtype=torch.float32).unsqueeze(1)
    edge_features = torch.exp(-((d - CENTERS) ** 2) / 0.1 ** 2)     #кожна відстань 26 чисел - розкладаємо відстань по діапазонах

    return torch.tensor(atoms), torch.tensor([sources, targets]), edge_features

In [68]:
def load_data(file_name):
    samples = []
    for row in csv.DictReader(open(file_name)):
        atoms, edges, edge_features = read_crystal(row["cif_file"])
        samples.append({
            "formula": row["formula"],
            "atoms": atoms,
            "edges": edges,
            "edge_features": edge_features,
            "pressure": float(row["pressure_GPa"]) /100.0,   # 100 ГПа -> 1.0
            "tc": float(row["Tc_K"]) / 100.0,                # 250 K   -> 2.5
        })
    print(f"Total structures: {len(samples)}")
    return samples

samples = load_data(DATA_FILE)

Total structures: 9


In [69]:
train_samples = []
test_samples = []
for s in samples:
    if s["formula"] in TEST_FORMULAS:
        test_samples.append(s)
    else:
        train_samples.append(s)

if len(test_samples) == 0:
    print("there are no compounds for checking")
    test_samples = train_samples

print(f"Structures in learning: {len(train_samples)}, in checking: {len(test_samples)}")
print("Train: ")
for s in train_samples:
    print(s['formula'], end=" ")
print()
print("Test: ")
for s in test_samples:
    print(s['formula'], end=" ")
print()

Structures in learning: 3, in checking: 6
Train: 
TbH6 TbH6 TiH2 
Test: 
CaH6 CaH6 H3Se H3Se H3Se H3Se 


In [70]:
class modelCGCNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.atom_table = torch.nn.Embedding(119, 64)   #кожному елементу — 64 числа
        self.conv1 = CGConv(64, dim=26, aggr="mean")
        self.conv2 = CGConv(64, dim=26, aggr="mean")
        self.conv3 = CGConv(64, dim=26, aggr="mean")
        self.output = torch.nn.Linear(65, 1)            #64 числа кристала + тиск -> Tc

    def forward(self, s):
        h = self.atom_table(s["atoms"])
        h = self.conv1(h, s["edges"], s["edge_features"])
        h = self.conv2(h, s["edges"], s["edge_features"])
        h = self.conv3(h, s["edges"], s["edge_features"])
        crystal = h.mean(dim=0)
        crystal = torch.cat([crystal, torch.tensor([s["pressure"]])])
        return self.output(crystal)[0]

model = modelCGCNN()

In [71]:
def train(model, samples):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for iteration in range(1, ITERATIONS + 1):
        random.shuffle(samples)
        total_error = 0
        for s in samples:
            optimizer.zero_grad()
            error = abs(model(s) - s["tc"])
            error.backward()
            optimizer.step()
            total_error += error.item()
        if iteration % 50 == 0:
            print(f"Iteration: {iteration}: avg error :{total_error / len(samples) * 100:.1f} K")

train(model, train_samples)

Iteration: 50: avg error :9.7 K
Iteration: 100: avg error :8.7 K
Iteration: 150: avg error :3.6 K
Iteration: 200: avg error :2.4 K


In [72]:
def test(model, samples):
    print("Compound    P, GPa   Tc fact   Tc predict")
    errors = []
    with torch.no_grad():
        for s in samples:
            tc_pred = model(s).item() * 100
            tc_true = s["tc"] * 100
            errors.append(abs(tc_pred - tc_true))
            print(f"{s['formula']:10s} {s['pressure'] * 100:6.0f} {tc_true:12.1f} {tc_pred:12.1f}")
    print(f"Avg error : {sum(errors) / len(errors):.1f} K")


test(model, test_samples)

Compound    P, GPa   Tc fact   Tc predict
CaH6          100        253.0        104.7
CaH6          135        236.0         99.8
H3Se          170        145.0        100.2
H3Se          200        113.0        101.1
H3Se          250        111.0        104.1
H3Se          300        110.0        105.3
Avg error : 58.8 K


In [73]:
mean_tc = sum(s["tc"] for s in train_samples) / len(train_samples)
baseline_errors = [abs(mean_tc - s["tc"]) * 100 for s in test_samples]
print(f"Predict 'always {mean_tc * 100:.2f} K': avg error {sum(baseline_errors) / len(baseline_errors):.2f} K")

Predict 'always 94.00 K': avg error 67.33 K


In [74]:
def predict(model, cif_file, pressure_GPa):
    atoms, edges, edge_features = read_crystal(cif_file)
    s = {"atoms": atoms, "edges": edges, "edge_features": edge_features, "pressure": pressure_GPa / 100}
    with torch.no_grad():
        return model(s).item() * 100


# tc = predict(model, "data/cif/------.cif",XXX)
# print(f"Прогноз Tc: {tc:.0f} K")